# 2. Gold standard

Aligns the lawyers' highlights to the sentences from `01_segmentation`, and measures agreement between the two lawyers.

**Reads:** `intermediate/sentences.pkl`, and `gold_annotations.csv` from `extract_annotations.ipynb` (one row per highlight, both annotators).
**Writes:** `intermediate/gold_sentences.csv`, one row per scored sentence with its five 0/1 labels and whether it is in the dev or test split, and `test_judgments.csv`, the held out judgments. Notebooks 03 to 06 evaluate against it.

## Setup

In [1]:
from argmine_common import *
import difflib
import html
from collections import defaultdict
from sklearn.metrics import cohen_kappa_score

GOLD_ANNOTATOR_COL, GOLD_FILE_COL, GOLD_TEXT_COL, GOLD_LABEL_COL = "annotator", "file", "text", "category"

# If a gold file cannot be linked to a judgment automatically, add it here: {"file name": doc_id}
FILE_TO_DOC_ID = {}

GOLD_MERGE = "union"           # two annotators on one judgment: "union" or "intersection"
ALIGN_MIN_COVER = 0.5          # share of a sentence that must fall inside a highlight
ALIGN_MIN_SPAN = 0.6           # share of a short highlight that must fall inside a sentence
N_TEST_DOCS = 3                # annotated judgments held out for the final test

corpus = load_corpus()
TITLE = titles(corpus)
sents = load_sentences()

## Load highlights and align them to sentences

Steps:
1. Load the combined highlight file and map labels to C1..C5. Labels that cannot be mapped (for example "unconfirmed_yellow") are reported and their sentences are excluded from scoring, rather than counted as negatives.
2. Link each file to a judgment: first by matching the file name to the judgment title, then by text overlap, then via `FILE_TO_DOC_ID`.
3. Align highlights to sentences. A sentence takes a highlight's label when at least `ALIGN_MIN_COVER` of it falls inside the highlight, when `ALIGN_MIN_SPAN` of a short highlight falls inside the sentence, or when the highlight sits wholly inside the sentence.
4. One Word file can contain highlights from more than one judgment (a quoted precedent). If a highlight does not match its file's judgment, we link that highlight by text overlap to another corpus judgment.

**Assumption to confirm with the lawyers:** highlights are exhaustive, so any reasoning sentence they left unhighlighted belongs to none of the five categories. If they only marked examples, recall scores below will be underestimated.

In [2]:
def norm_text(s):
    # Indian Kanoon HTML and PDF extraction both need this: quotes become &quot; / quot,
    # apostrophes become &#x27; / x27, and <a href> tags sit inside the legal text.
    s = html.unescape(str(s))
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\bx27\b", "'", s)
    s = re.sub(r"\bquot\b", " ", s)
    s = re.sub(r"(\w)-\s+(\w)", r"\1\2", s)
    s = re.sub(r"[^a-z0-9]+", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()

def norm_title(s):
    s = str(s)
    s = re.sub(r"\.(pdf|docx?|txt|gdoc)$", "", s, flags=re.I)
    return norm_text(s.replace("_", " "))


gold_spans = (pd.read_csv(GOLD_CSV)
              .rename(columns={GOLD_ANNOTATOR_COL: "annotator", GOLD_FILE_COL: "file",
                               GOLD_TEXT_COL: "text", GOLD_LABEL_COL: "raw_label"})
              [["annotator", "file", "raw_label", "text"]]
              .dropna(subset=["text"])
              .reset_index(drop=True))
gold_spans["annotator"] = gold_spans.annotator.str.lower()
gold_spans["label"] = gold_spans.raw_label.map(normalize_label)
gold_spans["span_id"] = range(len(gold_spans))

print(f"{len(gold_spans)} highlights from {gold_spans.file.nunique()} files")
print(pd.crosstab(gold_spans.raw_label.astype(str), gold_spans.label.fillna("UNMAPPED")).to_string())

94 highlights from 16 files
label                               C1  C2  C3  C4  C5
raw_label                                             
1_elements_offense                  33   0   0   0   0
2_balance_of_speech_and_reputation   0   2   0   0   0
3_procedural                         0   0  19   0   0
4_insufficient_evidence              0   0   0  23   0
5_abuse_of_process                   0   0   0   0  17


In [3]:
# ── Link files to judgments ───────────────────────────────────────────────────
title_index = {norm_title(t): d for d, t in TITLE.items()}

def shingles(s, n=6):
    toks = norm_text(s).split()
    return {" ".join(toks[i:i + n]) for i in range(max(len(toks) - n + 1, 0))}

shingle_index = defaultdict(set)
link_source = "full_text" if "full_text" in corpus.columns else "reason_text"
for d, t in zip(corpus.doc_id, corpus[link_source].fillna("")):
    for sh in shingles(t):
        shingle_index[sh].add(d)

def link_by_content(texts, min_share=0.3):
    hits, total = defaultdict(int), 0
    for t in texts:
        for sh in shingles(t):
            total += 1
            for d in shingle_index.get(sh, ()):
                hits[d] += 1
    if not hits or total == 0:
        return None, 0.0
    d, h = max(hits.items(), key=lambda x: x[1])
    return (d, h / total) if h / total >= min_share else (None, h / total)

links = []
for f, grp in gold_spans.groupby("file"):
    if f in FILE_TO_DOC_ID:
        links.append((f, FILE_TO_DOC_ID[f], "manual", 1.0)); continue
    d = title_index.get(norm_title(f))
    if d is None:
        close = difflib.get_close_matches(norm_title(f), list(title_index), n=1, cutoff=0.9)
        d = title_index[close[0]] if close else None
    if d is not None:
        links.append((f, d, "title", 1.0)); continue
    d, share = link_by_content(grp.text.tolist())
    links.append((f, d, "content" if d is not None else "UNLINKED", round(share, 2)))

link_df = pd.DataFrame(links, columns=["file", "doc_id", "how", "text_match"])
print(link_df.to_string(index=False))
gold_spans = gold_spans.merge(link_df[["file", "doc_id"]], on="file", how="left")

                                                                             file    doc_id     how  text_match
                              A.S.Sree Nandhini vs R.Uthaman on 27 November, 2014 155654015   title        1.00
                                                               Anurajan Jha, 2013 100113483 content        0.98
                      Asif_Iqbal_Naik_vs_State_Of_J_K_Ors_on_10_February_2023.PDF 131220563   title        1.00
                            Dr. Shashi Tharoor vs State & Anr. on 29 August, 2024 181209416   title        1.00
                    Jarnail Singh And Others vs Ranbir Singh on 22 November, 2013  68312913   title        1.00
                Komal Kumari vs The State(Nct Of Delhi) & Anr. on 20 August, 2024  48825553   title        1.00
                                     M.Chokkalingam vs A.P.Mani on 17 March, 2015  97845744   title        1.00
                           M.Nedunchezhian\tThe Bar Council Of Tamil Nadu\t2008\n 141782083 content     

In [4]:
# ── Align highlights to sentences ─────────────────────────────────────────────
def coverage(a, b):
    # share of string a that also appears in b
    if not a:
        return 0.0
    if a in b:
        return 1.0
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return sum(blk.size for blk in sm.get_matching_blocks() if blk.size >= 8) / len(a)

sents["norm"] = sents.text.map(norm_text)

# Word colour-runs arrive as tiny adjacent fragments. Join neighbours from the same
# file so a 7-word piece can still find its sentence; the original label is kept.
gold_spans = gold_spans.sort_values(["file", "span_id"]).reset_index(drop=True)
search, files, texts, nw = [], gold_spans.file.tolist(), gold_spans.text.astype(str).tolist(), []
nw = [len(t.split()) for t in texts]
for i, t in enumerate(texts):
    if nw[i] >= 20:
        search.append(t); continue
    chunk, j = [t], i - 1
    while j >= 0 and files[j] == files[i] and nw[j] < 20 and len(" ".join(chunk).split()) < 40:
        chunk.insert(0, texts[j]); j -= 1
    j = i + 1
    while j < len(texts) and files[j] == files[i] and nw[j] < 20 and len(" ".join(chunk).split()) < 40:
        chunk.append(texts[j]); j += 1
    search.append(" ".join(chunk))
gold_spans["search_text"] = search

def align_one(sp, doc_id):
    span_n = norm_text(getattr(sp, "search_text", None) or sp.text)
    own_n = norm_text(sp.text)
    span_tok = span_n.split()
    if len(span_tok) < 2:
        return []
    span_set = set(span_tok)
    rows = []
    cand = sents[sents.doc_id == doc_id]
    for sid, sent_n in zip(cand.index, cand.norm):
        tok = sent_n.split()
        if not tok:
            continue
        sent_in_span = coverage(sent_n, span_n)
        span_in_sent = coverage(span_n, sent_n)
        contained = span_n in sent_n
        if sent_in_span >= ALIGN_MIN_COVER or span_in_sent >= ALIGN_MIN_SPAN or contained:
            if own_n and own_n != span_n and coverage(own_n, sent_n) < 0.4 and own_n not in sent_n:
                continue
            rows.append({"sid": sid, "span_id": sp.span_id, "annotator": sp.annotator,
                         "label": sp.label, "cover": round(max(sent_in_span, span_in_sent), 2)})
    return rows

def best_other_doc(text, avoid):
    # Only long highlights vote: short fragments are too generic ("harm the reputation").
    if len(norm_text(text).split()) < 12:
        return None, 0.0
    d, share = link_by_content([text])
    return (d, share) if d is not None and d != avoid else (None, 0.0)

align_rows, first_hits = [], {}
for _, sp in gold_spans.dropna(subset=["doc_id"]).iterrows():
    rows = align_one(sp, sp.doc_id)
    if rows:
        first_hits[sp.span_id] = rows
        align_rows.extend(rows)

# One file can quote another judgment. Long unmatched highlights vote for a second doc;
# short fragments from the same file follow that vote if they then align.
relinked = []
full_norm = dict(zip(corpus.doc_id, corpus.get("full_text", corpus.reason_text).fillna("").map(norm_text)))

def in_original(sp):
    t, doc = norm_text(sp.text), sp.doc_id
    if len(t) < 20:
        return t in full_norm.get(doc, "")
    return t[:80] in full_norm.get(doc, "") or " ".join(t.split()[:12]) in full_norm.get(doc, "")

unhit = gold_spans[gold_spans.doc_id.notna() & ~gold_spans.span_id.isin(first_hits)]
unhit = unhit[~unhit.apply(in_original, axis=1)]   # still in this judgment, just not in reasoning/conclusion
for f, grp in unhit.groupby("file"):
    votes = []
    for _, sp in grp.iterrows():
        other, share = best_other_doc(sp.text, sp.doc_id)
        if other is not None:
            votes.append((other, share))
    if not votes:
        continue
    counts = pd.Series([d for d, _ in votes]).value_counts()
    winner = counts.index[0]
    if counts.iloc[0] < 2 and max(s for _, s in votes if _ == winner) < 0.8:
        continue
    for _, sp in grp.iterrows():
        rows = align_one(sp, winner)
        if rows:
            gold_spans.loc[gold_spans.span_id == sp.span_id, "doc_id"] = winner
            relinked.append((sp.span_id, int(sp.doc_id), int(winner), str(sp.text)[:80]))
            align_rows.extend(rows)
aligned = pd.DataFrame(align_rows, columns=["sid", "span_id", "annotator", "label", "cover"])
if relinked:
    print(f"Relinked {len(relinked)} highlights to a different judgment (quoted or misnamed source):")
    for sid, old, new, txt in relinked:
        print(f"  span {sid}: {old} -> {new}  {txt}")

# Last resort: pull only the matching sentence from full_text. Do not add the rest of the judgment.
def rescue_sentence(full, highlight):
    raw = re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", str(full)))).strip()
    needle = re.sub(r"\s+", " ", html.unescape(str(highlight))).strip()
    if len(needle) < 8:
        return None
    low, nlow = raw.lower(), needle.lower()
    i = low.find(nlow[:min(80, len(nlow))])
    if i < 0 and len(nlow) > 40:
        i = low.find(nlow[:40])
    if i < 0:
        return None
    left = raw.rfind(". ", 0, i)
    start = 0 if left < 0 else left + 2
    right = raw.find(". ", i + min(len(needle), 40))
    sent = raw[start:(len(raw) if right < 0 else right + 1)].strip()
    return sent if len(sent.split()) >= 4 else needle

hit_ids = set(r["span_id"] for r in align_rows)
rescued, new_rows = [], []
for _, sp in gold_spans.dropna(subset=["doc_id"]).iterrows():
    if sp.span_id in hit_ids:
        continue
    rec = corpus.loc[corpus.doc_id == sp.doc_id]
    if rec.empty:
        continue
    sent = rescue_sentence(rec.iloc[0].get("full_text", ""), sp.search_text if len(str(sp.search_text).split()) > len(str(sp.text).split()) else sp.text)
    if not sent:
        continue
    already = sents[(sents.doc_id == sp.doc_id) & (sents.norm == norm_text(sent))]
    if len(already):
        sid = already.index[0]
    else:
        sid = len(sents) + len(new_rows)
        new_rows.append({"doc_id": int(sp.doc_id), "para_idx": -1, "para_no": None, "sent_idx": 0,
                         "text": sent, "in_quote": False, "party_argument": False,
                         "seg_mode": "full_text_rescue", "section": "full_text_rescue",
                         "n_words": len(sent.split()), "norm": norm_text(sent)})
    align_rows.append({"sid": sid, "span_id": sp.span_id, "annotator": sp.annotator,
                       "label": sp.label, "cover": 1.0})
    rescued.append(sp.span_id)
if new_rows:
    sents = pd.concat([sents, pd.DataFrame(new_rows)], ignore_index=True)
    sents["norm"] = sents.text.map(norm_text)
    keep_cols = [c for c in sents.columns if c != "norm"]
    sents[keep_cols].to_pickle(SENTS_PKL)
    print(f"Added {len(new_rows)} full-text sentences for otherwise unplaced highlights (not the whole judgment).")
if rescued:
    print(f"Placed {len(rescued)} leftover highlights from full_text only.")

aligned = pd.DataFrame(align_rows, columns=["sid", "span_id", "annotator", "label", "cover"])
span_report = gold_spans.assign(n_sents=gold_spans.span_id.map(aligned.span_id.value_counts()).fillna(0).astype(int))
print(f"Highlights aligned to at least one sentence: {(span_report.n_sents > 0).mean():.0%}")
print(span_report.groupby("annotator").apply(lambda g: (g.n_sents > 0).mean()).round(2).to_string())

# Remaining unaligned highlights: other sections, HTML-only full_text, or text not in the corpus.
unaligned = span_report[(span_report.n_sents == 0) & span_report.doc_id.notna()]
if len(unaligned):
    full = dict(zip(corpus.doc_id, corpus.get("full_text", corpus.reason_text).fillna("").map(norm_text)))
    unaligned = unaligned.assign(in_full_text=[norm_text(t)[:80] in full.get(d, "")
                                               for t, d in zip(unaligned.text, unaligned.doc_id)])
    print(f"\n{len(unaligned)} unaligned highlights; found elsewhere in the full judgment: "
          f"{unaligned.in_full_text.sum()}")
    display(unaligned[["annotator", "file", "label", "in_full_text", "text"]].head(10))

Relinked 12 highlights to a different judgment (quoted or misnamed source):
  span 55: 155654015 -> 188509289  us, the mere telecasting of an imputation
  span 56: 155654015 -> 188509289  by itself may not constitute the offence of defamatio
  span 57: 155654015 -> 188509289  n unless such imputation has been made with the intention, notwithstanding that 
  span 58: 155654015 -> 188509289  harm the reputation of the person concerned.
  span 59: 155654015 -> 188509289  Further for attracting the ingredients of S.499 IPC, it is essential to plead th
  span 60: 155654015 -> 188509289  with such knowledge or consent to effect the reputation
  span 62: 155654015 -> 188509289  it was telecasted with their knowledge or consent or that they have
  span 63: 155654015 -> 188509289  perused or edited it with the intention to defame the 5th respondent.
  span 64: 155654015 -> 188509289  There are no specific pleading and averments in Annexure A1 complaint filed by t
  span 66: 155654015 -> 1885092

Added 14 full-text sentences for otherwise unplaced highlights (not the whole judgment).
Placed 14 leftover highlights from full_text only.
Highlights aligned to at least one sentence: 97%
annotator
priya     1.00
saanvi    0.96



3 unaligned highlights; found elsewhere in the full judgment: 1


,annotator,file,label,in_full_text,text
10,saanvi,"A.S.Sree Nandhini vs R.Uthaman on 27 November, 2014",C1,False,of the 2nd respondent herein.
14,saanvi,"A.S.Sree Nandhini vs R.Uthaman on 27 November, 2014",C4,False,he act with mala fide intention
16,saanvi,"A.S.Sree Nandhini vs R.Uthaman on 27 November, 2014",C3,True,he learned


In [5]:
# ── Gold label matrix over all sentences of the annotated judgments ───────────
gold_docs = sorted(aligned.merge(sents[["doc_id"]], left_on="sid", right_index=True).doc_id.unique())
G = sents[sents.doc_id.isin(gold_docs)].copy()

ann_by_doc = gold_spans.dropna(subset=["doc_id"]).groupby("doc_id").annotator.unique().map(set)
lab = aligned.dropna(subset=["label"]).groupby(["sid", "label"]).annotator.unique().map(set)

def gold_value(sid, cat, doc):
    who = lab.get((sid, cat), set())
    if GOLD_MERGE == "intersection" and len(ann_by_doc.get(doc, ())) > 1:
        return int(who >= ann_by_doc[doc])
    return int(len(who) > 0)

Y = np.array([[gold_value(s, c, d) for c in CATS] for s, d in zip(G.index, G.doc_id)])
unscored = set(aligned[aligned.label.isna()].sid)          # sentences under unmapped labels
keep = ~G.index.isin(unscored)
G, Y = G[keep], Y[keep]
DOCS = G.doc_id.values
GOLD_IDX = G.index.values          # row positions into `sents`

print(f"Gold set: {len(gold_docs)} judgments, {len(G)} sentences, {len(unscored)} excluded (unmapped labels)")
print(pd.DataFrame({"positive sentences": Y.sum(0),
                    "judgments with category": pd.DataFrame(Y).groupby(DOCS).max().sum().values},
                   index=[SHORT[c] for c in CATS]).to_string())

Gold set: 16 judgments, 527 sentences, 0 excluded (unmapped labels)
                        positive sentences  judgments with category
Ingredients/exceptions                  51                       11
Speech vs reputation                     5                        2
Procedural                              31                        5
Insufficient evidence                   27                        9
Abuse of process                        29                        8


## Hold out test judgments

`N_TEST_DOCS` annotated judgments are set aside as a test set. Whole judgments are held out, not sentences, because sentences of one judgment are alike and would leak between the splits. Among 500 random draws we keep the one whose judgments cover the most categories, so the test set is not missing a category by chance.

The draw is saved to `argument_mining/test_judgments.csv` and reused on every later run, so adding annotations does not reshuffle it. Delete that file to draw again, but only before anyone has looked at test scores.

In [6]:
if TEST_SPLIT_CSV.exists():
    TEST_DOCS = set(pd.read_csv(TEST_SPLIT_CSV).doc_id)
    gone = TEST_DOCS - set(gold_docs)
    if gone:
        print(f"Warning: test judgments no longer in the gold set: {sorted(gone)}. "
              f"Delete {TEST_SPLIT_CSV.name} to draw again.")
else:
    assert len(gold_docs) >= N_TEST_DOCS + 3, f"Only {len(gold_docs)} gold judgments: too few to hold out {N_TEST_DOCS}"
    doc_cats = pd.DataFrame(Y, columns=CATS).groupby(DOCS).max()
    rng = np.random.default_rng(SEED)
    draws = [rng.choice(doc_cats.index, N_TEST_DOCS, replace=False) for _ in range(500)]
    TEST_DOCS = set(max(draws, key=lambda d: doc_cats.loc[d].max().sum()))
    (pd.DataFrame({"doc_id": sorted(TEST_DOCS)}).assign(title=lambda t: t.doc_id.map(TITLE))
     .to_csv(TEST_SPLIT_CSV, index=False))
    print(f"Drew {len(TEST_DOCS)} test judgments and saved them to {TEST_SPLIT_CSV.relative_to(BASE)}")

TEST = np.isin(DOCS, list(TEST_DOCS))
for name, mask in (("dev", ~TEST), ("test", TEST)):
    print(f"\n{name}: {len(set(DOCS[mask]))} judgments, {mask.sum()} sentences")
    print(pd.Series(Y[mask].sum(0), index=[SHORT[c] for c in CATS], name="positive sentences").to_string())
for d in sorted(TEST_DOCS):
    print("test:", TITLE.get(d, d))


dev: 13 judgments, 460 sentences
Ingredients/exceptions    46
Speech vs reputation       5
Procedural                29
Insufficient evidence     25
Abuse of process          24

test: 3 judgments, 67 sentences
Ingredients/exceptions    5
Speech vs reputation      0
Procedural                2
Insufficient evidence     2
Abuse of process          5
test: Komal Kumari vs The State(Nct Of Delhi) & Anr. on 20 August, 2024
test: Rajeev Kumar Yadav vs State Of U.P. And Another on 4 December, 2019
test: Rahul Gandhi, M. P vs Rajesh Mahadev Kunte And Anr on 10 March, 2015


## Save

In [7]:
gold_table = (pd.DataFrame(Y, columns=CATS)
              .assign(sid=GOLD_IDX, doc_id=DOCS, split=np.where(TEST, "test", "dev"))
              [["sid", "doc_id", "split", *CATS]])
gold_table.to_csv(GOLD_SENTS_CSV, index=False)
print(f"Saved {len(gold_table)} gold sentences to {GOLD_SENTS_CSV.relative_to(BASE)}")

Saved 527 gold sentences to argument_mining/intermediate/gold_sentences.csv


## Agreement between the two lawyers

In [8]:
# ── Agreement between the two lawyers (judgments both annotated) ──────────────
shared = [d for d, who in ann_by_doc.items() if len(who) > 1 and d in gold_docs]
if shared:
    rows_s = np.isin(DOCS, shared)
    sids_s = GOLD_IDX[rows_s]
    annots = sorted(set.union(*[ann_by_doc[d] for d in shared]))[:2]
    kap = {}
    for c in CATS:
        a = [int(annots[0] in lab.get((s, c), set())) for s in sids_s]
        b = [int(annots[1] in lab.get((s, c), set())) for s in sids_s]
        kap[SHORT[c]] = round(cohen_kappa_score(a, b), 2) if (sum(a) + sum(b)) else np.nan
    print(f"Cohen's kappa, {annots[0]} vs {annots[1]}, {len(shared)} shared judgments, {rows_s.sum()} sentences")
    print(pd.Series(kap).to_string())
else:
    print("No judgment was annotated by both lawyers, so agreement cannot be computed yet. "
          "Double annotating 5 or more judgments would let us report it.")

No judgment was annotated by both lawyers, so agreement cannot be computed yet. Double annotating 5 or more judgments would let us report it.
